Make sure the right schema is used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

# Create bronze tables

In [0]:
CREATE TABLE IF NOT EXISTS customer_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS churn_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS log_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS ticket_bronze (
		ticket_id STRING,
		customer_id STRING,
		timestamp_created STRING,
		timestamp_closed STRING,
		subject STRING,
		description STRING,
		category STRING,
		priority STRING,
		channel STRING,
		status STRING,
		solved_in_hours STRING,
		log_id STRING,
		technical_issue_type STRING,
		ingestion_time TIMESTAMP
	);

In [0]:
CREATE TABLE IF NOT EXISTS agent_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS chat_bronze (
		session_id STRING,
		customer_id STRING,
		agent_id STRING,
		timestamp_start STRING,
		timestamp_end STRING,
		messages STRING,
		resolution_status STRING,
		log_id STRING,
		chat_reason STRING,
		ingestion_time TIMESTAMP
	)

# Fill tables

In [0]:
COPY INTO customer_bronze FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/customer/'
) FILEFORMAT = CSV FORMAT_OPTIONS (
	'header' = 'true',
	'multiLine' = 'true',
	'quote' = '"',
	'escape' = '"'
) COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
COPY INTO churn_bronze FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/churn'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
COPY INTO log_bronze FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/connection_log'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
COPY INTO agent_bronze FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/agent'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
SELECT
	*
FROM
	agent_bronze

agent_id,agent_name,employment_date,experience_level,monthly_salary_eur,ingestion_time
AGENT_001,Max Müller,2023-10-28,Junior,2766,2026-05-26T07:54:48.120Z
AGENT_002,Anna Schmidt,2023-12-12,Junior,3030,2026-05-26T07:54:48.120Z
AGENT_003,Lukas Schneider,2024-01-26,Mid,3682,2026-05-26T07:54:48.120Z
AGENT_004,Sophie Fischer,2024-03-11,Mid,3506,2026-05-26T07:54:48.120Z
AGENT_005,Tim Weber,2024-04-25,Mid,3581,2026-05-26T07:54:48.120Z
AGENT_006,Laura Meyer,2024-06-09,Junior,2947,2026-05-26T07:54:48.120Z
AGENT_007,Felix Wagner,2024-07-24,Mid,3690,2026-05-26T07:54:48.120Z
AGENT_008,Emma Becker,2024-09-07,Mid,3618,2026-05-26T07:54:48.120Z
AGENT_009,Paul Schulz,2024-10-22,Mid,3381,2026-05-26T07:54:48.120Z
AGENT_010,Mia Hoffmann,2024-12-06,Junior,3010,2026-05-26T07:54:48.120Z


In [0]:
%python

%pip install pandas openpyxl


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python

import os
import pandas
import glob
import openpyxl
from pyspark.sql.functions import current_timestamp

In [0]:
%python

path = "/Workspace/Users/nina.merkt@abat.de/databricks_demo/data"

csv_files = sorted(glob.glob(os.path.join(path, "ticket", "*.xlsx")), key=os.path.getmtime)

for f in csv_files:
  
    df_ticket = pandas.read_excel(f)

    print('Shape: ', df_ticket.shape)
    print('Columns: ', df_ticket.columns)
    print('Data Types: ', df_ticket.dtypes)

    df_ticket = df_ticket.astype(str)
    spark_df_ticket = spark.createDataFrame(df_ticket)

    spark_df_ticket = spark_df_ticket.withColumn("ingestion_time", current_timestamp())

    spark_df_ticket.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ticket_bronze")


In [0]:
%python

csv_files = sorted(glob.glob(os.path.join(path, "chat", "*.xlsx")), key=os.path.getmtime)

for f in csv_files:
  
    df_chat = pandas.read_excel(f)

    print('Shape: ', df_chat.shape)
    print('Columns: ', df_chat.columns)
    print('Data Types: ', df_chat.dtypes)

    df_chat = df_chat.astype(str)
    spark_df_chat = spark.createDataFrame(df_chat)

    spark_df_chat = spark_df_chat.withColumn("ingestion_time", current_timestamp())

    spark_df_chat.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("chat_bronze")

# Add table for PLZ to federal state
To test the accuracy of AI vs. more cumbersome classical method

In [0]:
CREATE TABLE IF NOT EXISTS plz_to_state

In [0]:
COPY INTO plz_to_state FROM (
	SELECT
		* FROM '/Volumes/sac/customer_service/cloud_storage/plz_list'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
88018,88018,0


# Add delete tables

Customer

In [0]:
CREATE TABLE IF NOT EXISTS customer_delete;

COPY INTO customer_delete FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/customer_delete'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
1,1,0


Agent

In [0]:
CREATE TABLE IF NOT EXISTS agent_delete;

COPY INTO agent_delete FROM (
	SELECT
		*,
		current_timestamp() AS ingestion_time FROM
	'/Volumes/sac/customer_service/cloud_storage/agent_delete'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
1,1,0
